[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-03.ipynb)

# 3주차 실습: 제주 공개 문서의 LoRA 미세조정

- 제공된 학습·평가 함수를 실행, 랭크 선택 한 곳만 변경하는 골격
- 기본 비교: r = 4와 r = 8을 각각 새 모델에서 실제 학습·평가
- 한국어 공개 데이터: 위키백과 「제주특별자치도」 고정 판 42464087
- 과제: 보이는 텍스트에 자연·지형 키워드가 포함됐는지의 교수 규칙 기반 proxy 분류
- 감성 분류와 사람의 주제 gold가 아님. 규칙을 얼마나 모방했는지의 작은 동작 확인
- CPU, 최대 입력 64토큰, 각 설정 6 step, 배치 2. 데이터 8개 학습·4개 평가
- 전체 미세조정은 파라미터 수만 계산. 전체 학습의 시간·정확도·peak 메모리 비교는 미실행


## 0. 준비

- 모델과 공개 원문은 실행 때 다운로드, 실패를 가짜 데이터로 대체하지 않는 원칙
- torch CPU 스레드 2, Trainer도 CPU 고정. GPU 의존 없음


In [ ]:
%pip -q install transformers datasets peft accelerate


## 1. 제주 원문과 한국어 tokenizer

- 원문: https://ko.wikipedia.org/w/index.php?oldid=42464087
- 저자: 해당 판의 역사에 등재된 위키백과 기여자, CC BY-SA 4.0
- 원문 문단에서 최대 52토큰 조각을 추출, 특별 토큰을 포함해 최대 64토큰 입력
- proxy label 1: 보이는 조각에 기후·지형·화산·해안·한라산 중 하나 포함
- proxy label 0: 해당 문자열 없음. 실제 주제의 부정이나 감정의 부정을 의미하지 않음
- 동일 원문 문단이 학습·평가에 동시에 들어가지 않도록 문단 ID 기준 분리


In [ ]:
import gc, html, json, random, re, time, urllib.parse, urllib.request
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model

torch.set_num_threads(2)
SEED = 42
MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
MAX_LENGTH = 64
STEPS = 6
BATCH_SIZE = 2
OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
TERMS = ("기후", "지형", "화산", "해안", "한라산")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
url = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
request = urllib.request.Request(url, headers={"User-Agent": "deepnlp-course/1.0 (yj.lee@chu.ac.kr)"})
with urllib.request.urlopen(request, timeout=60) as response:
    parsed = json.load(response)["parse"]
assert parsed["revid"] == OLDID, "수집 판 번호 불일치"

def clean(fragment):
    fragment = re.sub(r"<sup\b[^>]*>.*?</sup>", "", fragment, flags=re.S)
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", fragment))).strip()

paragraphs = [clean(p) for p in re.findall(r"<p\b[^>]*>(.*?)</p>", parsed["text"], flags=re.S)]
paragraphs = [p for p in paragraphs if len(p) >= 40]
examples = []
for pid, paragraph in enumerate(paragraphs):
    ids = tokenizer(paragraph, add_special_tokens=False)["input_ids"]
    for start in range(0, len(ids), 52):
        window = ids[start:start + 52]
        if len(window) < 8:
            continue
        # 공개 문자열 API로 재토큰화, 4.x/5.x 호환. label은 실제 모델 입력 기준
        visible_text = tokenizer.decode(window, skip_special_tokens=True)
        encoded = tokenizer(visible_text, add_special_tokens=True, padding="max_length",
                            truncation=True, max_length=MAX_LENGTH)
        visible_text = tokenizer.decode(encoded["input_ids"], skip_special_tokens=True)
        label = int(any(term in visible_text for term in TERMS))
        examples.append({"paragraph_id": f"P{pid:03d}", "chunk_id": f"P{pid:03d}-T{start}",
                         "visible_text": visible_text, "labels": label,
                         "input_ids": encoded["input_ids"], "attention_mask": encoded["attention_mask"]})
print("공개 출처:", SOURCE, "라이선스: CC BY-SA 4.0")
print("원문 문단:", len(paragraphs), "사용 가능 조각:", len(examples))
print("규칙 기반 label:", TERMS, "포함=1 / 없음=0")


## 2. 작은 균형 학습·평가셋

- 각 label에서 서로 다른 문단 6개 선택: 4개 학습, 2개 평가
- 한 문단에서 한 조각만 사용, 모든 설정에서 같은 분할 사용
- 평가 4건이므로 한 건 차이는 정확도 0.25 차이. 일반 성능 추정에 부족한 규모


In [ ]:
rng = random.Random(SEED)
shuffled = examples.copy()
rng.shuffle(shuffled)
used_paragraphs, train_rows, eval_rows = set(), [], []
# 키워드 포함 문단을 먼저 확보, 두 클래스 사이에서도 원문 문단 중복 금지
for label in (1, 0):
    chosen = []
    for row in shuffled:
        if row["labels"] == label and row["paragraph_id"] not in used_paragraphs:
            chosen.append(row)
            used_paragraphs.add(row["paragraph_id"])
            if len(chosen) == 6:
                break
    assert len(chosen) == 6, f"label {label}의 독립 문단 부족: 원문·tokenizer·규칙 확인"
    train_rows.extend(chosen[:4])
    eval_rows.extend(chosen[4:])
assert not {r["paragraph_id"] for r in train_rows}.intersection(r["paragraph_id"] for r in eval_rows)
rng.shuffle(train_rows)
rng.shuffle(eval_rows)
columns = ("input_ids", "attention_mask", "labels")
train_ds = Dataset.from_list([{k: row[k] for k in columns} for row in train_rows])
eval_ds = Dataset.from_list([{k: row[k] for k in columns} for row in eval_rows])
print("학습:", len(train_ds), "평가:", len(eval_ds))
print("평가 문단:", [r["paragraph_id"] for r in eval_rows])
for row in eval_rows:
    print(row["chunk_id"], "proxy label", row["labels"], row["visible_text"][:100])


## 3. 제공된 LoRA trainer

- 설정마다 같은 seed로 한국어 사전학습 모델과 분류 head를 새로 초기화
- query·value에 LoRA, 분류 head도 학습. rank 외 조건은 동일
- alpha = 2 x rank로 alpha/r = 2 유지. 학생이 바꾸는 값은 rank 한 곳
- 초기/학습 후 평가, 6 step 학습 시간, 파라미터 수와 텐서 bytes 출력
- 텐서 bytes는 파라미터 저장량. 활성값·gradient·optimizer 상태·peak RSS는 미측정
- 시간은 trainer.train() 범위, 다운로드·모델 로딩·평가는 제외


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"proxy_accuracy": float(np.mean(predictions == labels))}

def run_lora(rank):
    set_seed(SEED)
    base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, torch_dtype=torch.float32).to("cpu")
    full_count = sum(p.numel() for p in base.parameters())
    full_bytes = sum(p.numel() * p.element_size() for p in base.parameters())
    config = LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.1, bias="none",
                        task_type="SEQ_CLS", target_modules=["query", "value"])
    adapted = get_peft_model(base, config)
    trainable = sum(p.numel() for p in adapted.parameters() if p.requires_grad)
    adapter_count = sum(p.numel() for name, p in adapted.named_parameters() if p.requires_grad and "lora_" in name)
    head_count = trainable - adapter_count
    parameter_bytes = sum(p.numel() * p.element_size() for p in adapted.parameters())
    trainable_bytes = sum(p.numel() * p.element_size() for p in adapted.parameters() if p.requires_grad)
    args = TrainingArguments(output_dir=f"./week03-r{rank}", use_cpu=True, max_steps=STEPS,
                             per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                             learning_rate=5e-4, seed=SEED, data_seed=SEED, report_to=[],
                             save_strategy="no", logging_strategy="no", eval_strategy="no",
                             dataloader_num_workers=0, dataloader_pin_memory=False, label_names=["labels"], disable_tqdm=True)
    trainer = Trainer(model=adapted, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                      compute_metrics=compute_metrics)
    before = trainer.evaluate()
    start = time.perf_counter()
    trainer.train()
    train_seconds = time.perf_counter() - start
    after = trainer.evaluate()
    prediction = trainer.predict(eval_ds).predictions.argmax(axis=-1).tolist()
    result = {"rank": rank, "alpha": 2 * rank, "steps": int(trainer.state.global_step),
              "full_model_parameters_count_only": full_count, "full_model_parameter_bytes_only": full_bytes,
              "trainable_parameters": trainable, "adapter_parameters": adapter_count,
              "head_parameters": head_count, "trainable_parameter_bytes": trainable_bytes,
              "all_parameter_bytes": parameter_bytes, "train_seconds": train_seconds,
              "proxy_accuracy_before": before["eval_proxy_accuracy"],
              "proxy_accuracy_after": after["eval_proxy_accuracy"], "eval_predictions": prediction,
              "full_finetuning_training": "미측정", "peak_memory": "미측정"}
    print(json.dumps(result, ensure_ascii=False, indent=2))
    del trainer, adapted, base
    gc.collect()
    return result


## 4. 먼저 r = 4 기준선 실제 학습·평가

- 모델 파라미터를 세는 것만으로 학습했다고 말하지 않는 원칙
- 아래 함수에서 실제 trainer.train()과 평가 실행


In [ ]:
baseline = run_lora(4)


## 5. 랭크 한 곳 선택하고 실제 재학습

- 아래 값 한 곳만 변경. 기본 r = 8도 같은 함수로 실제 학습·평가
- 모든 셀 기본 실행으로 r = 4와 r = 8 비교 완료
- rank를 다른 값으로 바꾸고 이 셀부터 다시 실행해 비교


In [ ]:
# TODO: 비교 랭크 하나만 변경 (기본 8, 예: 4 또는 16)
rank = 8


In [ ]:
changed = run_lora(rank)
print("\n설정 | 학습 파라미터 | proxy 정확도 전→후 | 학습 초 | 파라미터 MiB")
for result in (baseline, changed):
    print(f"r={result['rank']} | {result['trainable_parameters']:,} | "
          f"{result['proxy_accuracy_before']:.2f}→{result['proxy_accuracy_after']:.2f} | "
          f"{result['train_seconds']:.2f} | {result['all_parameter_bytes'] / 2**20:.2f}")
print("전체 미세조정: 파라미터 count만 계산, 학습 시간·정확도 미측정")
report = {"source": SOURCE, "revision": OLDID, "license": "CC BY-SA 4.0", "label_type": "keyword proxy",
          "label_terms": list(TERMS), "model": MODEL_NAME, "seed": SEED, "max_length": MAX_LENGTH,
          "train_chunks": [r["chunk_id"] for r in train_rows], "eval_chunks": [r["chunk_id"] for r in eval_rows],
          "eval_labels": [r["labels"] for r in eval_rows], "baseline": baseline, "changed": changed}
Path("week03-results.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("실행 결과: week03-results.json")


## 6. 결과 해석과 과제 1

1. r = 4와 비교 rank에서 어댑터 파라미터는 얼마나 늘었는가? 분류 head까지 합친 학습 파라미터가 정확히 두 배가 아닌 이유 설명
2. proxy 평가 4개에서 한 건 차이는 정확도 몇 차이인가? 이 작은 숫자를 제주 전체 문서 분류 성능으로 일반화할 수 없는 이유 설명
3. 각 설정의 실제 학습 시간·파라미터 저장 bytes 기록. peak 메모리와 전체 미세조정의 학습 성능은 미측정으로 보존
4. alpha/r을 유지한 이유와, 데이터·step·seed·배치를 고정한 이유 설명
5. 제공된 비교표에 자신의 결과와 단위를 기록하고 도내 소규모 조직의 상정 예산 안에서 선택 근거 제시

- 데이터에 대한 대규모 성능 실험이 아니라 제공 PEFT 골격의 실제 학습·평가 흐름 확인
- 과제 1 상세: assignments/week-03/README.md
- 제출 학번 폴더에는 이름·연락처 미포함. 공개 원문 전체와 다운로드 캐시는 커밋하지 않는 원칙
